# CLIP-ReID: PyTorch → ONNX → TensorRT

Ноутбук выполняет полный пайплайн конвертации:

```
MSMT17_clipreid_ViT-B-16_60.pth
         │
         ▼  torch.onnx.export
  clipreid_msmt17.onnx
         │
         ▼  TensorRT builder
  clipreid_msmt17.trt
```

**Требования:**
- NVIDIA GPU (обязательно — TRT и хардкод `.cuda()` в модели)
- CUDA ≥ 11.x
- TensorRT ≥ 8.x
- Python пакеты: `torch torchvision onnx onnxruntime-gpu tensorrt pycuda`

**Параметры модели (ViT-B-16, MSMT17):**
| Параметр | Значение |
|---|---|
| Вход | `[B, 3, 256, 128]` float32 |
| Выход | `[B, 1280]` float32, L2-нормализован |
| camera_num | 15 |
| num_class | 1041 |

In [ ]:
# Установка зависимостей (выполнить один раз)
# !pip install onnx onnxruntime-gpu
# !pip install tensorrt pycuda
# TensorRT рекомендуется устанавливать через NVIDIA wheel или conda, а не pip

In [ ]:
import os
import sys
import time
import numpy as np
import torch
import torch.nn as nn

# ── Добавляем clip-reid-msmt17 в путь поиска модулей ──────────────────────────
REPO_DIR = os.path.dirname(os.path.abspath("__file__"))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Проверяем CUDA ────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "CUDA недоступна! Модель CLIP-ReID использует .cuda() в __init__ — "
    "для экспорта и TRT нужен GPU."
)
print(f"CUDA доступна: {torch.cuda.get_device_name(0)}")
print(f"torch       : {torch.__version__}")

## Шаг 0: Настройте пути

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║               НАСТРОЙТЕ ЭТИ ПУТИ ПОД ВАШЕ ОКРУЖЕНИЕ            ║
# ╚══════════════════════════════════════════════════════════════════╝

# Путь к файлу конфига (vit_clipreid.yml из папки clip-reid)
CONFIG_PATH  = "../clip-reid/configs/person/vit_clipreid.yml"

# Путь к обученным весам (.pth)
WEIGHTS_PATH = "../clip-reid/MSMT17_clipreid_ViT-B-16_60.pth"

# Куда сохранить выходные файлы
ONNX_PATH     = "clipreid_msmt17.onnx"
TRT_PATH      = "clipreid_msmt17.trt"       # TRT FP16
TRT_INT8_PATH = "clipreid_msmt17_int8.trt"  # TRT INT8 (с калибровкой)

# Датасет для INT8 калибровки (часть train MSMT17)
DATASET_ROOT    = "../../MSMT17_V1"          # корень датасета
INT8_CALIB_IMGS = 1000                       # кол-во изображений для калибровки

# Параметры модели MSMT17
CAMERA_NUM   = 15
NUM_CLASS    = 1041

# Параметры TRT FP16 engine
TRT_PRECISION = "fp16"   # fp32 | fp16
TRT_MIN_BATCH = 1
TRT_OPT_BATCH = 8
TRT_MAX_BATCH = 32

print(f"Config     : {os.path.abspath(CONFIG_PATH)}")
print(f"Weights    : {os.path.abspath(WEIGHTS_PATH)}")
print(f"ONNX       : {os.path.abspath(ONNX_PATH)}")
print(f"TRT FP16   : {os.path.abspath(TRT_PATH)}")
print(f"TRT INT8   : {os.path.abspath(TRT_INT8_PATH)}")
print(f"Calib imgs : {INT8_CALIB_IMGS}")

assert os.path.exists(CONFIG_PATH),  f"Конфиг не найден: {CONFIG_PATH}"
assert os.path.exists(WEIGHTS_PATH), f"Веса не найдены: {WEIGHTS_PATH}"
print("\nПути проверены ✓")

## Шаг 1: Экспорт PyTorch → ONNX

Модель `build_transformer` в `eval()` возвращает:
```python
torch.cat([img_feature, img_feature_proj], dim=1)  # [B, 768+512] = [B, 1280]
```
Обёртка `CLIPReIDForONNX` фиксирует `cam_label`/`view_label` как константы
(ONNX не поддерживает управляющие аргументы динамически) и L2-нормализует выход.

In [ ]:
class CLIPReIDForONNX(nn.Module):
    """
    Тонкая обёртка для ONNX-экспорта CLIP-ReID (без SIE).

    Передаём cam_label=None, view_label=None — модель уходит в ветку
    else: cv_embed = None и не обращается к self.cv_embed,
    которого нет при SIE_CAMERA=False / SIE_VIEW=False.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, 3, 256, 128] float32 (CLIP нормализация)
        Returns:
            embedding: [B, 1280] float32, L2-нормализован
        """
        # cam_label=None → forward уходит в ветку else: cv_embed = None
        feat = self.model(x, cam_label=None, view_label=None)

        if isinstance(feat, (tuple, list)):
            feat = feat[-1]

        feat = feat.float()   # на случай float16 из CLIP
        feat = feat / feat.norm(dim=-1, keepdim=True)  # L2-норм
        return feat

In [ ]:
from config import cfg
from model.make_model_clipreid import make_model

# 1. Загружаем конфиг
print("[1/4] Загружаем конфиг...")
cfg.defrost()  # на случай повторного запуска ячейки
cfg.merge_from_file(CONFIG_PATH)
cfg.freeze()
print(f"      backbone  : {cfg.MODEL.NAME}")
print(f"      neck_feat : {cfg.TEST.NECK_FEAT}")
print(f"      SIE_CAMERA: {cfg.MODEL.SIE_CAMERA}")
print(f"      SIE_VIEW  : {cfg.MODEL.SIE_VIEW}")
print(f"      input     : {cfg.INPUT.SIZE_TRAIN}")

# 2. Строим модель
print("\n[2/4] Строим модель...")
base_model = make_model(cfg, num_class=NUM_CLASS,
                        camera_num=CAMERA_NUM, view_num=0)
base_model.load_param(WEIGHTS_PATH)
base_model.eval()
base_model = base_model.cuda()
print("      Модель загружена ✓")

# 3. Тестовый прогон — cam_label=None, иначе AttributeError при SIE_CAMERA=False
print("\n[3/4] Проверяем выход модели...")
with torch.no_grad():
    dummy_cuda = torch.randn(2, 3, 256, 128).cuda()
    raw_out    = base_model(dummy_cuda, cam_label=None, view_label=None)
    if isinstance(raw_out, (tuple, list)):
        raw_out = raw_out[-1]
print(f"      Размерность выхода модели: {raw_out.shape}")

EMB_DIM = raw_out.shape[-1]
print(f"      EMB_DIM = {EMB_DIM}")

# 4. Оборачиваем и экспортируем
print("\n[4/4] Экспортируем в ONNX...")
onnx_wrapper = CLIPReIDForONNX(base_model)
onnx_wrapper.eval()

dummy_export = torch.randn(1, 3, 256, 128).cuda()

torch.onnx.export(
    onnx_wrapper,
    dummy_export,
    ONNX_PATH,
    input_names=["image"],
    output_names=["embedding"],
    dynamic_axes={
        "image":     {0: "batch_size"},
        "embedding": {0: "batch_size"},
    },
    opset_version=14,
    do_constant_folding=True,
    training=torch.onnx.TrainingMode.EVAL,
    verbose=False,
)

size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f"      Сохранён: {ONNX_PATH} ({size_mb:.1f} MB) ✓")

In [ ]:
# ── Верификация ONNX ────────────────────────────────────────────────────────
import onnx
import onnxruntime as ort

print("Верификация ONNX графа...")
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("  Граф валиден ✓")

# Показываем I/O
for inp in onnx_model.graph.input:
    shape = [d.dim_value if d.dim_value > 0 else d.dim_param
             for d in inp.type.tensor_type.shape.dim]
    print(f"  Input : {inp.name} {shape}")
for out in onnx_model.graph.output:
    shape = [d.dim_value if d.dim_value > 0 else d.dim_param
             for d in out.type.tensor_type.shape.dim]
    print(f"  Output: {out.name} {shape}")

# Сравниваем PyTorch vs ONNX Runtime
print("\nСравнение PyTorch ↔ ONNX Runtime...")
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
sess = ort.InferenceSession(ONNX_PATH, providers=providers)
print(f"  ORT провайдер: {sess.get_providers()[0]}")

test_input = torch.randn(2, 3, 256, 128)

with torch.no_grad():
    pt_out = onnx_wrapper(test_input.cuda()).cpu().numpy()

ort_out = sess.run(["embedding"], {"image": test_input.numpy()})[0]

diff    = np.max(np.abs(pt_out - ort_out))
cos_sim = float(np.mean([np.dot(pt_out[i], ort_out[i]) for i in range(len(pt_out))]))

print(f"  Макс. отклонение : {diff:.2e}")
print(f"  Cosine sim       : {cos_sim:.8f}  (должно быть ~1.0)")

if diff < 1e-3:
    print("  ✓ ONNX корректен — можно конвертировать в TRT")
else:
    print("  ⚠ Большое отклонение — проверьте модель")

## Шаг 2: ONNX → TensorRT Engine

TRT engine **привязан к конкретному GPU и версии TRT**. При смене GPU пересобирайте.

| Режим | Скорость | Качество | Требования |
|---|---|---|---|
| `fp32` | базовая | максимальное | — |
| `fp16` | ~2x быстрее | потеря < 0.1% | Tensor Cores (Volta+) |
| `int8` | ~4x быстрее | потеря ~0.5% | калибровочный датасет |

In [ ]:
try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit
    TRT_AVAILABLE = True
    print(f"TensorRT версия : {trt.__version__}")
except ImportError as e:
    TRT_AVAILABLE = False
    print(f"TensorRT не установлен: {e}")
    print("Установите: pip install tensorrt pycuda")
    print("Или через NVIDIA: https://developer.nvidia.com/tensorrt")

TRT_LOGGER = trt.Logger(trt.Logger.WARNING) if TRT_AVAILABLE else None

In [ ]:
def build_trt_engine(
    onnx_path:   str,
    engine_path: str,
    precision:   str = "fp16",
    min_batch:   int = 1,
    opt_batch:   int = 8,
    max_batch:   int = 32,
) -> None:
    """
    Конвертирует ONNX → TensorRT serialized engine (.trt).

    Args:
        onnx_path   : путь к .onnx файлу
        engine_path : путь для записи .trt файла
        precision   : fp32 | fp16 | int8
        min_batch   : минимальный batch_size (для dynamic shape профиля)
        opt_batch   : оптимальный batch_size (TRT оптимизирует под него)
        max_batch   : максимальный batch_size
    """
    assert TRT_AVAILABLE, "TensorRT не установлен"

    print(f"\n{'='*60}")
    print(f"СБОРКА TensorRT ENGINE")
    print(f"  ONNX      : {onnx_path}")
    print(f"  Output    : {engine_path}")
    print(f"  Precision : {precision.upper()}")
    print(f"  Batch     : min={min_batch}  opt={opt_batch}  max={max_batch}")
    print(f"{'='*60}")

    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, TRT_LOGGER)
    config = builder.create_builder_config()

    # 2 GB рабочей памяти для оптимизатора
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)

    # ── Режим точности ────────────────────────────────────────────────────
    if precision == "fp16":
        if builder.platform_has_fast_fp16:
            config.set_flag(trt.BuilderFlag.FP16)
            print("  FP16 включён ✓")
        else:
            print("  ⚠ GPU не поддерживает быстрый FP16, используем FP32")

    elif precision == "int8":
        if builder.platform_has_fast_int8:
            config.set_flag(trt.BuilderFlag.INT8)
            config.int8_calibrator = _make_random_calibrator()
            print("  INT8 включён ✓ (случайный калибровщик)")
        else:
            config.set_flag(trt.BuilderFlag.FP16)
            print("  ⚠ INT8 не поддерживается, используем FP16")

    # ── Парсим ONNX ───────────────────────────────────────────────────────
    print("\nПарсим ONNX...")
    with open(onnx_path, "rb") as f:
        ok = parser.parse(f.read())
    if not ok:
        for i in range(parser.num_errors):
            print(f"  Ошибка: {parser.get_error(i)}")
        raise RuntimeError("Не удалось распарсить ONNX")
    print(f"  Нод в сети: {network.num_layers}")

    # ── Dynamic shapes ────────────────────────────────────────────────────
    print("Настраиваем dynamic shapes...")
    profile = builder.create_optimization_profile()
    profile.set_shape(
        "image",
        (min_batch, 3, 256, 128),
        (opt_batch, 3, 256, 128),
        (max_batch, 3, 256, 128),
    )
    config.add_optimization_profile(profile)

    # ── Сборка ────────────────────────────────────────────────────────────
    print("Собираем engine (1–10 минут на первом запуске)...")
    t0 = time.time()
    serialized = builder.build_serialized_network(network, config)

    if serialized is None:
        raise RuntimeError("Не удалось собрать TensorRT engine")

    with open(engine_path, "wb") as f:
        f.write(serialized)

    elapsed  = time.time() - t0
    size_mb  = os.path.getsize(engine_path) / 1024 / 1024
    print(f"  Готово за {elapsed:.1f} сек")
    print(f"  Сохранён: {engine_path} ({size_mb:.1f} MB) ✓")


def _make_random_calibrator():
    """Случайный INT8 калибровщик. Для лучшего качества замените на реальные данные."""
    class _Calibrator(trt.IInt8EntropyCalibrator2):
        def __init__(self, n_batches=10, batch_size=8):
            super().__init__()
            self.n_batches  = n_batches
            self.batch_size = batch_size
            self.current    = 0
            self.buf = cuda.mem_alloc(batch_size * 3 * 256 * 128 * 4)

        def get_batch_size(self): return self.batch_size

        def get_batch(self, names):
            if self.current >= self.n_batches: return None
            data = np.random.randn(self.batch_size, 3, 256, 128).astype(np.float32)
            cuda.memcpy_htod(self.buf, data.ravel())
            self.current += 1
            return [int(self.buf)]

        def read_calibration_cache(self):   return None
        def write_calibration_cache(self, c): pass

    return _Calibrator()


# ── Запускаем сборку ──────────────────────────────────────────────────────────
if TRT_AVAILABLE:
    build_trt_engine(
        onnx_path=ONNX_PATH,
        engine_path=TRT_PATH,
        precision=TRT_PRECISION,
        min_batch=TRT_MIN_BATCH,
        opt_batch=TRT_OPT_BATCH,
        max_batch=TRT_MAX_BATCH,
    )
else:
    print("Пропускаем — TRT не доступен")

## Шаг 3: Верификация TRT Engine

Сравниваем выходы **TensorRT** и **ONNX Runtime** на одинаковом входе.
Допустимое отклонение: `< 1e-2` для fp16, `< 1e-4` для fp32.

In [ ]:
def _get_max_shape(engine, name, mode, batch_size) -> tuple:
    """
    Безопасно читает максимальную форму тензора из TRT engine.
    Три уровня fallback — см. комментарий в benchmark_compare_models.ipynb.
    """
    try:
        raw  = engine.get_tensor_profile_shape(name, 0)
        dims = raw[2]

        if hasattr(dims, 'nbDims') and dims.nbDims > 0:
            return tuple(int(dims[j]) for j in range(dims.nbDims))

        result = []
        for j in range(8):
            try:
                result.append(int(dims[j]))
            except Exception:
                break
        if result:
            return tuple(result)
    except Exception:
        pass

    import tensorrt as trt
    if mode == trt.TensorIOMode.INPUT:
        return (batch_size, 3, 256, 128)
    else:
        return (batch_size, 1280)


class TRTSession:
    """
    Обёртка для инференса через TensorRT engine.

    Использование:
        with TRTSession("model.trt") as sess:
            emb = sess.infer(img_np)   # [B, 3, 256, 128] → [B, EMB_DIM]
    """

    def __init__(self, engine_path: str):
        runtime = trt.Runtime(TRT_LOGGER)
        with open(engine_path, "rb") as f:
            self.engine  = runtime.deserialize_cuda_engine(f.read())
        self.context = self.engine.create_execution_context()
        self.stream  = cuda.Stream()

        self._inputs  = []
        self._outputs = []

        for i in range(self.engine.num_io_tensors):
            name  = self.engine.get_tensor_name(i)
            dtype = trt.nptype(self.engine.get_tensor_dtype(name))
            mode  = self.engine.get_tensor_mode(name)
            max_shape = _get_max_shape(self.engine, name, mode, batch_size=32)
            size      = int(np.prod(max_shape))
            h_mem = cuda.pagelocked_empty(size, dtype)
            d_mem = cuda.mem_alloc(h_mem.nbytes)
            entry = {"name": name, "host": h_mem, "device": d_mem,
                     "dtype": dtype, "max_shape": max_shape}
            if mode == trt.TensorIOMode.INPUT:
                self._inputs.append(entry)
            else:
                self._outputs.append(entry)

        print(f"TRT engine загружен ✓")
        print(f"  Input  (max): {self._inputs[0]['max_shape']}")
        print(f"  Output (max): {self._outputs[0]['max_shape']}")

    def infer(self, image_np: np.ndarray) -> np.ndarray:
        batch = image_np.shape[0]
        self.context.set_input_shape("image", image_np.shape)
        np.copyto(self._inputs[0]["host"][:image_np.size], image_np.ravel())
        cuda.memcpy_htod_async(self._inputs[0]["device"],
                               self._inputs[0]["host"], self.stream)
        self.context.execute_async_v3(stream_handle=self.stream.handle)
        cuda.memcpy_dtoh_async(self._outputs[0]["host"],
                               self._outputs[0]["device"], self.stream)
        self.stream.synchronize()
        emb_dim = self._outputs[0]["max_shape"][-1]
        return self._outputs[0]["host"][:batch * emb_dim].reshape(batch, emb_dim).copy()

    def destroy(self):
        del self.context
        del self.engine
        for buf in self._inputs + self._outputs:
            buf["device"].free()

    def __enter__(self):  return self
    def __exit__(self, *a): self.destroy()

In [ ]:
if not TRT_AVAILABLE:
    print("TRT не доступен — пропускаем")
else:
    print("Верификация TRT engine...")
    print("=" * 60)

    # ORT сессия для сравнения
    ort_sess = ort.InferenceSession(
        ONNX_PATH,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    )

    test_img = np.random.randn(2, 3, 256, 128).astype(np.float32)

    # ONNX Runtime выход
    ort_out = ort_sess.run(["embedding"], {"image": test_img})[0]

    # TRT выход
    with TRTSession(TRT_PATH) as trt_sess:
        trt_out = trt_sess.infer(test_img)

    diff    = np.max(np.abs(ort_out - trt_out))
    cos_sim = np.mean([np.dot(ort_out[i], trt_out[i]) for i in range(len(ort_out))])

    tol = 1e-2 if TRT_PRECISION in ("fp16", "int8") else 1e-4

    print(f"\n  Форматы: ONNX={ort_out.shape} | TRT={trt_out.shape}")
    print(f"  Макс. отклонение TRT↔ORT : {diff:.2e}  (допуск: {tol:.0e})")
    print(f"  Cosine similarity        : {cos_sim:.8f}")
    print(f"  L2 норма TRT             : {np.linalg.norm(trt_out[0]):.6f}  (должна быть 1.0)")

    if diff < tol:
        print("\n  ✓ TRT engine корректен!")
    else:
        print("\n  ⚠ Отклонение больше порога — проверьте точность")

## Шаг 4: Бенчмарк — TRT vs ONNX Runtime

In [ ]:
def benchmark(trt_path: str, ort_session, n_warmup: int = 20, n_runs: int = 200):
    """Сравнивает латентность TRT и ONNX Runtime по батч-размерам."""
    print(f"\n{'='*65}")
    print(f"{'БЕНЧМАРК':^65}")
    print(f"  Прогрев={n_warmup}  Замеры={n_runs}")
    print(f"{'='*65}")
    print(f"  {'batch':>6}  {'TRT ms':>9}  {'ORT ms':>9}  {'ms/img TRT':>11}  {'speedup':>8}")
    print(f"  {'-'*6}  {'-'*9}  {'-'*9}  {'-'*11}  {'-'*8}")

    with TRTSession(trt_path) as trt_sess:
        for bs in [1, 2, 4, 8, 16, 32]:
            if bs > TRT_MAX_BATCH:
                continue
            dummy = np.random.randn(bs, 3, 256, 128).astype(np.float32)

            # Прогрев TRT
            for _ in range(n_warmup):
                trt_sess.infer(dummy)
            t_trt = []
            for _ in range(n_runs):
                t0 = time.perf_counter()
                trt_sess.infer(dummy)
                t_trt.append((time.perf_counter() - t0) * 1000)

            # Прогрев ORT
            for _ in range(n_warmup):
                ort_session.run(["embedding"], {"image": dummy})
            t_ort = []
            for _ in range(n_runs):
                t0 = time.perf_counter()
                ort_session.run(["embedding"], {"image": dummy})
                t_ort.append((time.perf_counter() - t0) * 1000)

            trt_ms  = np.mean(t_trt)
            ort_ms  = np.mean(t_ort)
            speedup = ort_ms / trt_ms
            print(f"  {bs:>6}  {trt_ms:>9.2f}  {ort_ms:>9.2f}  {trt_ms/bs:>11.2f}  {speedup:>8.2f}x")

    print(f"{'='*65}")


if TRT_AVAILABLE:
    benchmark(TRT_PATH, ort_sess)
else:
    print("TRT не доступен — пропускаем")

## Шаг 5: TRT INT8 Build

INT8 требует **калибровочный датасет** — реальные изображения, на которых модель будет работать в production. TRT измеряет диапазон активаций и подбирает INT8 scale-факторы.

| Параметр | Рекомендация |
|---|---|
| `INT8_CALIB_IMGS` | 500–2000 (больше — незначительный прирост) |
| Тип данных | Реальные изображения из MSMT17 train (не синтетика) |
| Кэш | `int8_calib.cache` — повторный запуск загружает кэш и пропускает калибровку |

> **Ожидаемый прирост mAP:** FP32 ≈ FP16 > INT8 (потеря ~0.3–0.8%)  
> **Ожидаемый прирост скорости:** INT8 ~3–4× быстрее FP32 на V100

In [ ]:
import torchvision.transforms as T
from PIL import Image
import glob
import random

# ──────────────────────────────────────────────────────────────────────────────
# MSMT17 INT8 Calibrator
# Использует реальные изображения из train MSMT17 для точной калибровки.
# Случайная калибровка (_make_random_calibrator) хуже на ~0.5–1% mAP.
# ──────────────────────────────────────────────────────────────────────────────
class MSMT17Int8Calibrator(trt.IInt8EntropyCalibrator2):
    """
    INT8 калибровщик на основе реальных train-изображений MSMT17.

    Рекомендуемое кол-во изображений: 500–2000.
    Слишком мало (<100) → шум в scale-факторах.
    Слишком много (>5000) → медленная сборка без выигрыша.
    """

    TRANSFORM = T.Compose([
        T.Resize((256, 128), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(
            mean=[0.48145466, 0.4578275,  0.40821073],
            std= [0.26862954, 0.26130258, 0.27577711],
        ),
    ])

    def __init__(self, dataset_root: str, n_images: int = 1000,
                 batch_size: int = 8, cache_file: str = "int8_calib.cache"):
        super().__init__()
        self.batch_size  = batch_size
        self.cache_file  = cache_file
        self.current_idx = 0

        # Собираем пути из train MSMT17
        train_dir = os.path.join(dataset_root, "train")
        list_path = os.path.join(dataset_root, "list_train.txt")

        if os.path.exists(list_path):
            with open(list_path) as f:
                paths = [os.path.join(train_dir, line.split()[0])
                         for line in f if line.strip()]
        else:
            paths = glob.glob(os.path.join(train_dir, "**", "*.jpg"), recursive=True)
            paths += glob.glob(os.path.join(train_dir, "**", "*.png"), recursive=True)

        random.seed(42)
        random.shuffle(paths)
        self.paths = paths[:n_images]

        # Предзагружаем все батчи в CPU RAM
        print(f"  Загружаем {len(self.paths)} калибровочных изображений...")
        self.batches = []
        for i in range(0, len(self.paths), batch_size):
            batch_paths = self.paths[i:i + batch_size]
            imgs = []
            for p in batch_paths:
                try:
                    img = Image.open(p).convert("RGB")
                    imgs.append(self.TRANSFORM(img))
                except Exception:
                    continue
            if imgs:
                tensor = torch.stack(imgs).numpy().astype(np.float32)
                self.batches.append(tensor)

        # GPU буфер под максимальный батч
        max_size = max(b.shape[0] for b in self.batches) * 3 * 256 * 128
        self._gpu_buf = cuda.mem_alloc(max_size * 4)
        print(f"  Калибровочных батчей: {len(self.batches)}")

    def get_batch_size(self):
        return self.batch_size

    def get_batch(self, names):
        if self.current_idx >= len(self.batches):
            return None
        batch = self.batches[self.current_idx]
        self.current_idx += 1
        cuda.memcpy_htod(self._gpu_buf, batch.ravel())
        return [int(self._gpu_buf)]

    def read_calibration_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, "rb") as f:
                data = f.read()
            print(f"  Загружен кэш калибровки: {self.cache_file}")
            return data
        return None

    def write_calibration_cache(self, cache):
        with open(self.cache_file, "wb") as f:
            f.write(cache)
        print(f"  Кэш калибровки сохранён: {self.cache_file}")


# ── Сборка TRT INT8 engine ────────────────────────────────────────────────────
def build_trt_int8(onnx_path, engine_path, dataset_root, n_images,
                   min_batch=1, opt_batch=8, max_batch=32):
    """Собирает TRT INT8 с реальной калибровкой на MSMT17 train."""
    assert TRT_AVAILABLE, "TRT не установлен"

    print(f"\n{'='*60}")
    print(f"СБОРКА TensorRT INT8 ENGINE")
    print(f"  ONNX   : {onnx_path}")
    print(f"  Output : {engine_path}")
    print(f"  Calib  : {n_images} изображений из {dataset_root}")
    print(f"  Batch  : min={min_batch}  opt={opt_batch}  max={max_batch}")
    print(f"{'='*60}")

    calibrator = MSMT17Int8Calibrator(
        dataset_root=dataset_root,
        n_images=n_images,
        batch_size=opt_batch,
    )

    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, TRT_LOGGER)
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)
    config.set_flag(trt.BuilderFlag.INT8)
    if builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)  # FP16 как fallback для слоёв без INT8
    config.int8_calibrator = calibrator
    print("  INT8 + FP16 fallback включены ✓")

    with open(onnx_path, "rb") as f:
        parser.parse(f.read())

    profile = builder.create_optimization_profile()
    profile.set_shape(
        "image",
        (min_batch, 3, 256, 128),
        (opt_batch, 3, 256, 128),
        (max_batch, 3, 256, 128),
    )
    config.add_optimization_profile(profile)

    print("\nСобираем INT8 engine (может занять 5–15 минут)...")
    t0 = time.time()
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError("Не удалось собрать TRT INT8 engine")

    with open(engine_path, "wb") as f:
        f.write(serialized)

    elapsed = time.time() - t0
    size_mb = os.path.getsize(engine_path) / 1024 / 1024
    print(f"  Готово за {elapsed:.1f} сек")
    print(f"  Сохранён: {engine_path} ({size_mb:.1f} MB) ✓")


if TRT_AVAILABLE:
    build_trt_int8(
        onnx_path=ONNX_PATH,
        engine_path=TRT_INT8_PATH,
        dataset_root=DATASET_ROOT,
        n_images=INT8_CALIB_IMGS,
        min_batch=TRT_MIN_BATCH,
        opt_batch=TRT_OPT_BATCH,
        max_batch=TRT_MAX_BATCH,
    )
else:
    print("Пропускаем — TRT не доступен")

In [ ]:
if not TRT_AVAILABLE:
    print("TRT не доступен — пропускаем")
else:
    print("Верификация TRT INT8 engine...")
    print("=" * 60)

    ort_sess_v = ort.InferenceSession(
        ONNX_PATH,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    )

    test_img = np.random.randn(2, 3, 256, 128).astype(np.float32)
    ort_out  = ort_sess_v.run(["embedding"], {"image": test_img})[0]

    with TRTSession(TRT_INT8_PATH) as trt_sess:
        int8_out = trt_sess.infer(test_img)

    diff    = np.max(np.abs(ort_out - int8_out))
    cos_sim = np.mean([np.dot(ort_out[i], int8_out[i]) for i in range(len(ort_out))])

    print(f"\n  Макс. отклонение INT8↔ORT : {diff:.2e}  (допуск: 5e-2)")
    print(f"  Cosine similarity         : {cos_sim:.8f}  (ожидается > 0.999)")
    print(f"  L2 норма                  : {np.linalg.norm(int8_out[0]):.6f}  (должна быть 1.0)")

    if diff < 0.05:
        print("\n  ✓ TRT INT8 engine корректен!")
    else:
        print("\n  ⚠ Большое отклонение — попробуйте увеличить INT8_CALIB_IMGS")

## Итог

После успешного выполнения у вас будет:

| Файл | Назначение |
|---|---|
| `clipreid_msmt17.onnx` | Переносимый формат, работает везде через ONNXRuntime |
| `clipreid_msmt17.trt` | Максимальная скорость на вашем конкретном GPU |

### Использование TRT engine в production

```python
with TRTSession("clipreid_msmt17.trt") as sess:
    img = preprocess_image("person.jpg")      # [1, 3, 256, 128] float32
    embedding = sess.infer(img)               # [1, 1280] float32, L2-норм
```

### Если нужно пересобрать engine
При обновлении TRT, смене GPU или изменении batch range — просто перезапустите **Шаг 2**.